# 01 — Definição do problema e preparação dos dados

**T326 – Ciência dos Dados · Projeto 2 – Aprendizado de Máquina: Olist** · Prof. Caio Ponte

## 1. Definição do problema

**Contexto da Olist.** A Olist é uma empresa brasileira que conecta pequenos e médios lojistas aos grandes
marketplaces (Mercado Livre, Americanas, etc.) por meio de um único contrato. O lojista vende, e o produto
é enviado diretamente ao cliente pelos parceiros logísticos. O dataset público tem ~100 mil pedidos feitos
entre 2016 e 2018 em todo o Brasil.

**Problema de regressão.**
- **Amostra:** cada linha é **um pedido**.
- **Variável alvo:** `delivery_time_days` = (data de entrega ao cliente − data da compra), em dias (com fração).
- **Entradas:** características do pedido conhecidas **no momento da compra** (valor, frete, peso, dimensões,
  categoria, UF do cliente, distância vendedor→cliente, dia/hora da compra).

**Desafio real e impacto.** O prazo informado pela Olist é muito conservador (em média ~24 dias para entregas
que levam ~12) e, mesmo assim, parte dos pedidos chega atrasada. Atraso derruba a nota de avaliação do cliente
(mostrado abaixo), aumenta custo de atendimento e prejudica a reputação dos lojistas. Prever melhor o tempo
de entrega permite prometer prazos mais realistas e identificar pedidos com risco de demora.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import olist_features as of

TAB = ROOT / "resultados/tabelas"; TAB.mkdir(parents=True, exist_ok=True)
FIG = ROOT / "resultados/graficos"; FIG.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

## 2. Tabelas do dataset e como se relacionam

O dataset tem 9 tabelas (CSV). Relações principais (chaves):

```
customers ──(customer_id)── orders ──(order_id)── order_items ──(product_id)── products
    │                                                  └──────(seller_id)──── sellers
    └──(zip_code_prefix)── geolocation ──(zip_code_prefix)──────────────────────┘
order_payments, order_reviews ──(order_id)── orders
```

**Tabelas utilizadas e justificativa**

| Tabela | Uso | Por quê |
|---|---|---|
| orders | alvo + data da compra | define a amostra (pedido) e as datas de compra/entrega |
| order_items | valor, frete, nº de itens/vendedores | o que foi comprado e de quantos vendedores |
| products | peso, dimensões, categoria | produtos maiores/pesados demoram mais |
| sellers | UF/CEP do vendedor | origem do envio |
| customers | UF/CEP do cliente | destino do envio |
| geolocation | lat/long por CEP | distância vendedor→cliente (Haversine) |
| order_reviews | **só para mostrar o impacto** | não entra no modelo (é posterior à entrega) |
| order_payments | não usada | não influencia a logística |

In [2]:
tables = of.load_tables()
inventario = pd.DataFrame([{"tabela": k, "linhas": len(v), "colunas": v.shape[1]} for k, v in tables.items()])
inventario.to_csv(TAB / "01_tabelas.csv", index=False)
inventario

,tabela,linhas,colunas
0,orders,99441,8
1,items,112650,7
2,products,32951,9
3,sellers,3095,4
4,customers,99441,5
5,geolocation,1000163,5
6,payments,103886,5
7,reviews,99224,7
8,translation,71,2


## 3. Impacto do atraso na satisfação do cliente

In [3]:
raw = of.build_features(tables)
entregues = raw[raw.order_status.eq("delivered") & raw[of.TARGET].notna()].copy()
entregues["atrasado"] = entregues[of.TARGET] > entregues["olist_estimated_days"]
notas = tables["reviews"].groupby("order_id").review_score.mean()
entregues["nota"] = entregues.order_id.map(notas)
impacto = pd.DataFrame({
    "indicador": ["Tempo real médio (dias)", "Prazo estimado pela Olist (dias)", "% de pedidos atrasados",
                  "Nota média – no prazo", "Nota média – atrasado"],
    "valor": [entregues[of.TARGET].mean(), entregues.olist_estimated_days.mean(), entregues.atrasado.mean() * 100,
              entregues.loc[~entregues.atrasado, "nota"].mean(), entregues.loc[entregues.atrasado, "nota"].mean()]})
impacto.to_csv(TAB / "01_impacto.csv", index=False)
impacto

,indicador,valor
0,Tempo real médio (dias),12.56
1,Prazo estimado pela Olist (dias),23.74
2,% de pedidos atrasados,8.11
3,Nota média – no prazo,4.29
4,Nota média – atrasado,2.57


## 4. Engenharia de atributos

Construção (em `src/olist_features.py`):
1. Coordenada de cada CEP = mediana das lat/long da `geolocation` (descartando pontos fora do Brasil).
2. Junção `order_items` + `products` + `sellers` + cliente — validando cardinalidade (nenhuma linha duplicada).
3. **Agregação por pedido** (um pedido pode ter vários itens): somas, médias, máximos, contagens.
4. Distância Haversine vendedor→cliente por item → média/mín/máx por pedido.
5. Atributos de calendário a partir da data da compra.

Nenhuma informação posterior à compra (aprovação, envio, entrega, avaliação) entra como feature.

In [4]:
features = of.feature_summary()
features.to_csv(TAB / "01_features.csv", index=False)
print(len(features), "features:", (features.tipo == "numérica").sum(), "numéricas e", (features.tipo == "categórica").sum(), "categóricas")
features

30 features: 26 numéricas e 4 categóricas


,feature,tipo,origem,descrição
0,products_value_total,numérica,order_items,Soma dos preços dos itens (R$)
1,freight_value_total,numérica,order_items,Soma do frete dos itens (R$)
2,order_value_total,numérica,order_items,Produtos + frete (R$)
3,items_count,numérica,order_items,Número de itens do pedido
4,products_count,numérica,order_items,Produtos distintos no pedido
5,sellers_count,numérica,order_items,Vendedores distintos no pedido
6,seller_states_count,numérica,sellers,UFs distintas dos vendedores
7,same_state_items_fraction,numérica,sellers + customers,Fração de itens com vendedor na UF do cliente
8,weight_g_total,numérica,products,Peso total dos itens (g)
9,weight_g_mean,numérica,products,Peso médio por item (g)


## 5. Filtragem e limpeza

- Somente pedidos **entregues** com data de entrega (o alvo precisa existir).
- Pedidos sem itens ou com tempo ≤ 0 seriam inconsistências.
- Medidas inválidas (peso/dimensões ≤ 0, preço/frete negativo) viram valores ausentes.
- **Ruído:** entregas acima de 60 dias (0,3%) são tratadas como anomalias (extravio, erro de registro).

In [5]:
data, funil = of.clean(raw)
funil.to_csv(TAB / "01_funil_limpeza.csv", index=False)
funil

,etapa,pedidos_restantes,removidos
0,Pedidos originais,99441,0
1,Status 'delivered',96478,2963
2,Com data de entrega,96470,8
3,Com itens,96470,0
4,Tempo de entrega > 0,96470,0
5,Entrega ≤ 60 dias (remove ruído),96164,306


## 6. Valores faltantes (tratados no pipeline do notebook 03 com `SimpleImputer`)

In [6]:
faltantes = data[of.FEATURES].isna().sum().rename("faltantes").to_frame()
faltantes["%"] = faltantes.faltantes / len(data) * 100
faltantes = faltantes[faltantes.faltantes > 0].sort_values("faltantes", ascending=False)
faltantes.to_csv(TAB / "01_faltantes.csv")
faltantes

,faltantes,%
main_category,1328,1.38
distance_km_max,474,0.49
distance_km_mean,474,0.49
distance_km_min,474,0.49
weight_g_mean,22,0.02
weight_g_total,22,0.02
width_cm_max,16,0.02
height_cm_max,16,0.02
length_cm_max,16,0.02
volume_cm3_total,16,0.02


In [7]:
of.DATASET_PATH.parent.mkdir(parents=True, exist_ok=True)
data.to_parquet(of.DATASET_PATH, index=False)
print(f"Dataset salvo: {of.DATASET_PATH.relative_to(ROOT)} — {data.shape[0]:,} pedidos × {len(of.FEATURES)} features + alvo")

Dataset salvo: dados\tratados\olist_dataset.parquet — 96,164 pedidos × 30 features + alvo
